In [1]:
# Single-cell LangGraph + DuckDB "Hospital Assistant" demo
# If needed, this will try to install deps. (Comment out if you manage deps yourself.)

import sys, subprocess, os, re, json
from datetime import datetime
from zoneinfo import ZoneInfo
from typing import TypedDict, List, Dict, Any, Optional

def _pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# ---- dependencies (auto-install if missing) ----
try:
    import duckdb
except ImportError:
    _pip_install(["duckdb"])
    import duckdb

try:
    from langgraph.graph import StateGraph, END
except ImportError:
    _pip_install(["langgraph"])
    from langgraph.graph import StateGraph, END


# =========================
# 1) DuckDB connection
# =========================
DB_PATH = ":memory:"  # <-- set to "hospital.duckdb" for your real file
con = duckdb.connect(DB_PATH)

def today_iso(tz="America/New_York") -> str:
    return datetime.now(ZoneInfo(tz)).date().isoformat()

def db_query(sql: str, params: Optional[list] = None) -> List[Dict[str, Any]]:
    """Minimal DB query tool: returns list[dict] rows, or [{"error": "..."}]."""
    try:
        cur = con.execute(sql, params or [])
        cols = [d[0] for d in cur.description] if cur.description else []
        rows = cur.fetchall() if cols else []
        return [dict(zip(cols, r)) for r in rows]
    except Exception as e:
        return [{"error": str(e), "sql": sql}]

def seed_demo_data():
    # Demo schema (skip/adjust if you already have your own schema)
    con.execute("""
        CREATE TABLE IF NOT EXISTS patients (
            patient_id INTEGER PRIMARY KEY,
            name TEXT,
            dob DATE
        );
    """)
    con.execute("""
        CREATE TABLE IF NOT EXISTS doctors (
            doctor_id INTEGER PRIMARY KEY,
            name TEXT,
            specialty TEXT
        );
    """)
    con.execute("""
        CREATE TABLE IF NOT EXISTS calendar_meetings (
            meeting_id INTEGER PRIMARY KEY,
            patient_id INTEGER,
            doctor_id INTEGER,
            meeting_date DATE,
            meeting_time TEXT,
            location TEXT,
            notes TEXT
        );
    """)

    # Seed only if empty
    if con.execute("SELECT COUNT(*) FROM patients;").fetchone()[0] == 0:
        con.executemany(
            "INSERT INTO patients VALUES (?, ?, ?);",
            [
                (123, "Alex Rivera", "1985-04-12"),
                (124, "Jamie Chen", "1992-09-30"),
            ],
        )

    if con.execute("SELECT COUNT(*) FROM doctors;").fetchone()[0] == 0:
        con.executemany(
            "INSERT INTO doctors VALUES (?, ?, ?);",
            [
                (1, "Dr. Patel", "Cardiology"),
                (2, "Dr. Nguyen", "Primary Care"),
            ],
        )

    # Ensure at least one meeting exists for patient 123 today (for demo)
    t = today_iso()
    exists = con.execute(
        "SELECT COUNT(*) FROM calendar_meetings WHERE patient_id=? AND meeting_date=?;",
        [123, t],
    ).fetchone()[0]
    if exists == 0:
        next_id = con.execute("SELECT COALESCE(MAX(meeting_id), 0) + 1 FROM calendar_meetings;").fetchone()[0]
        con.execute(
            """
            INSERT INTO calendar_meetings
              (meeting_id, patient_id, doctor_id, meeting_date, meeting_time, location, notes)
            VALUES
              (?, ?, ?, ?, ?, ?, ?);
            """,
            [next_id, 123, 2, t, "10:00", "Room 3B", "Follow-up consultation"],
        )

seed_demo_data()

# Tool: fetch meeting details for a patient + day
def fetch_meeting_details(patient_id: int, day: str) -> List[Dict[str, Any]]:
    # Adjust joins/columns to match your real schema if needed
    return db_query(
        """
        SELECT
            m.meeting_id,
            m.meeting_date,
            m.meeting_time,
            m.location,
            m.notes,
            m.patient_id,
            p.name AS patient_name,
            m.doctor_id,
            d.name AS doctor_name,
            d.specialty AS doctor_specialty
        FROM calendar_meetings m
        LEFT JOIN patients p ON p.patient_id = m.patient_id
        LEFT JOIN doctors  d ON d.doctor_id  = m.doctor_id
        WHERE m.patient_id = ? AND m.meeting_date = ?
        ORDER BY m.meeting_time;
        """,
        [patient_id, day],
    )

# =========================
# 2) LangGraph state + nodes
# =========================
class AgentState(TypedDict):
    history: List[Dict[str, str]]
    user_input: str
    intent: str
    parsed: Dict[str, Any]
    ui_recommendation: Optional[Dict[str, Any]]
    tool_result: Optional[Any]
    assistant_output: str



In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# --- Add these imports near the top of your cell ---
import json
from openai import OpenAI

# --- Create the OpenAI client once (reads OPENAI_API_KEY automatically) ---
oa_client = OpenAI()

# --- JSON Schema for your intent parsing output ---
INTENT_SCHEMA = {
    "type": "object",
    "properties": {
        "intent": {
            "type": "string",
            "enum": ["get_meeting_details", "list_patients", "view_patient", "list_doctors", "unknown"]
        },
        # Use 0 when no patient id is present (keeps schema simple/reliable)
        "patient_id": {"type": "integer", "minimum": 0},
        # Use "" when day is not applicable
        "day": {"type": "string"},
        "ui_recommendation": {
            "type": "object",
            "properties": {
                "view": {"type": "string", "enum": ["Doctor", "Patient", "Calendar"]},
                "action": {"type": "string", "enum": ["view", "list", "add", "insert"]},
                "params": {"type": "object"}
            },
            "required": ["view", "action", "params"],
            "additionalProperties": False
        }
    },
    "required": ["intent", "patient_id", "day", "ui_recommendation"],
    "additionalProperties": False
}

def llm_parse_intent(user_text: str, today_str: str) -> dict:
    """
    Returns dict matching INTENT_SCHEMA exactly:
      { intent, patient_id, day, ui_recommendation }
    """
    system = f"""
You are an intent router for a hospital app UI.

UI Views: Doctor, Patient, Calendar
Actions: view, list, add, insert

Today's date (local): {today_str}

Rules:
- If user asks about a meeting/appointment/visit with a patient id, choose intent=get_meeting_details,
  set patient_id, set day to today's date unless the user specifies a different date.
  Recommend ui_recommendation.view="Calendar", action="view", params={{"patientId": <id>, "day": <YYYY-MM-DD>}}.
- If user says "list patients", choose list_patients and recommend Patient/list.
- If user says "view patient 123", choose view_patient and recommend Patient/view with patientId.
- If user says "list doctors", choose list_doctors and recommend Doctor/list.
- Otherwise intent=unknown and ui_recommendation can be a no-op like Patient/view with empty params.

Return ONLY valid JSON that matches the provided schema.
""".strip()

    # Responses API + Structured Outputs (JSON Schema)
    resp = oa_client.responses.create(
        model="gpt-4.1",
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "hospital_intent",
                "strict": True,
                "schema": INTENT_SCHEMA
            }
        },
    )

    # With strict schema, output_text should be valid JSON
    return json.loads(resp.output_text)

def parse_intent_node(state):
    text = (state.get("user_input") or "").strip()
    day = today_iso()  # your existing helper (America/New_York)

    # LLM parse with a safe fallback
    try:
        parsed = llm_parse_intent(text, day)
        intent = parsed["intent"]

        # Normalize into your graph state fields
        state["intent"] = intent
        state["parsed"] = {}
        state["ui_recommendation"] = parsed["ui_recommendation"]

        if intent == "get_meeting_details":
            state["parsed"] = {
                "patient_id": int(parsed["patient_id"]),
                "day": parsed["day"] or day
            }
        elif intent == "view_patient":
            state["parsed"] = {"patient_id": int(parsed["patient_id"])}
        else:
            state["parsed"] = {}

        return state

    except Exception as e:
        # Fallback: keep your previous regex logic or a minimal default
        state["intent"] = "unknown"
        state["parsed"] = {}
        state["ui_recommendation"] = {"view": "Patient", "action": "view", "params": {}}
        state["tool_result"] = [{"error": f"LLM parse failed: {str(e)}"}]
        return state


In [ ]:
#def parse_intent_node(state: AgentState) -> AgentState:
#    text = (state.get("user_input") or "").strip()
#    day = today_iso()
#
#    intent = "help"
#    parsed: Dict[str, Any] = {}
#    ui: Optional[Dict[str, Any]] = None
#
#    # Meeting query pattern: look for "meeting"/"appointment" and "patient <digits>"
#    m = re.search(r"\bpatient\s*#?\s*(\d+)\b", text, re.IGNORECASE)
#    if m and re.search(r"\b(meeting|appointment|visit)\b", text, re.IGNORECASE):
#        pid = int(m.group(1))
#        intent = "get_meeting_details"
#        parsed = {"patient_id": pid, "day": day}
#        ui = {"view": "Calendar", "action": "view", "params": {"patientId": pid, "day": day}}
#
#    # A couple extra examples for your 3-view UI:
#    elif re.search(r"\blist\b.*\bpatients?\b", text, re.IGNORECASE):
#        intent = "list_patients"
#        ui = {"view": "Patient", "action": "list", "params": {}}
#
#    elif m and re.search(r"\bview\b.*\bpatient\b", text, re.IGNORECASE):
#        pid = int(m.group(1))
#        intent = "view_patient"
#        parsed = {"patient_id": pid}
#        ui = {"view": "Patient", "action": "view", "params": {"patientId": pid}}
#
#    elif re.search(r"\blist\b.*\bdoctors?\b", text, re.IGNORECASE):
#        intent = "list_doctors"
#        ui = {"view": "Doctor", "action": "list", "params": {}}
#
#    state["intent"] = intent
#    state["parsed"] = parsed
#    state["ui_recommendation"] = ui
#    return state

def run_tool_node(state: AgentState) -> AgentState:
    intent = state["intent"]
    parsed = state.get("parsed") or {}

    if intent == "get_meeting_details":
        state["tool_result"] = fetch_meeting_details(parsed["patient_id"], parsed["day"])

    elif intent == "list_patients":
        state["tool_result"] = db_query("SELECT patient_id, name, dob FROM patients ORDER BY patient_id LIMIT 50;")

    elif intent == "view_patient":
        pid = parsed["patient_id"]
        state["tool_result"] = db_query("SELECT * FROM patients WHERE patient_id = ?;", [pid])

    elif intent == "list_doctors":
        state["tool_result"] = db_query("SELECT doctor_id, name, specialty FROM doctors ORDER BY doctor_id LIMIT 50;")

    else:
        state["tool_result"] = None

    return state

def format_response_node(state: AgentState) -> AgentState:
    intent = state["intent"]
    ui = state.get("ui_recommendation")
    tool = state.get("tool_result")

    if intent == "get_meeting_details":
        pid = state["parsed"]["patient_id"]
        day = state["parsed"]["day"]

        if not tool:
            out = f"I couldn't find any meetings for patient {pid} on {day}."
        elif isinstance(tool, list) and tool and "error" in tool[0]:
            out = f"DB error while fetching meetings: {tool[0]['error']}"
        else:
            lines = []
            for r in tool:
                lines.append(
                    f"- {r['meeting_time']} — {r.get('patient_name','(unknown patient)')} with "
                    f"{r.get('doctor_name','(unknown doctor)')} ({r.get('doctor_specialty','')}); "
                    f"{r.get('location','')}. Notes: {r.get('notes','')}"
                )
            out = f"Meeting details for patient {pid} on {day}:\n" + "\n".join(lines)

    elif intent == "list_patients":
        out = "Patients (first 50):\n" + "\n".join(
            [f"- {r['patient_id']}: {r['name']} (DOB {r['dob']})" for r in (tool or [])]
        )

    elif intent == "view_patient":
        out = "Patient record:\n" + json.dumps(tool or [], indent=2)

    elif intent == "list_doctors":
        out = "Doctors (first 50):\n" + "\n".join(
            [f"- {r['doctor_id']}: {r['name']} ({r['specialty']})" for r in (tool or [])]
        )

    else:
        out = (
            "Try one of these:\n"
            "  • list patients\n"
            "  • view patient 123\n"
            "  • i'm having a meeting with patient 123, what are the meeting details?\n"
        )

    if ui:
        # This is the payload you can pass to your UI layer
        out += "\n\nUI_RECOMMENDATION=" + json.dumps(ui, indent=2)

    state["assistant_output"] = out
    state["history"] = (state.get("history") or []) + [
        {"role": "user", "content": state["user_input"]},
        {"role": "assistant", "content": out},
    ]
    return state

def route_after_parse(state: AgentState) -> str:
    if state["intent"] in {"get_meeting_details", "list_patients", "view_patient", "list_doctors"}:
        return "run_tool"
    return "format_response"

# Build graph
g = StateGraph(AgentState)
g.add_node("parse_intent", parse_intent_node)
g.add_node("run_tool", run_tool_node)
g.add_node("format_response", format_response_node)

g.set_entry_point("parse_intent")
g.add_conditional_edges("parse_intent", route_after_parse, {"run_tool": "run_tool", "format_response": "format_response"})
g.add_edge("run_tool", "format_response")
g.add_edge("format_response", END)

app = g.compile()

# =========================
# 3) Simple REPL
# =========================
state: AgentState = {
    "history": [],
    "user_input": "",
    "intent": "",
    "parsed": {},
    "ui_recommendation": None,
    "tool_result": None,
    "assistant_output": "",
}

print("Hospital LangGraph agent ready. Type 'quit' to exit.")
while True:
    print("*" * 100)
    user = input("\nYou: ").strip()
    if user.lower() in {"quit", "exit"}:
        break
    state["user_input"] = user
    state = app.invoke(state)
    print("\nAssistant:\n" + state["assistant_output"])


Hospital LangGraph agent ready. Type 'quit' to exit.
****************************************************************************************************



You:  are u ther?



Assistant:
Try one of these:
  • list patients
  • view patient 123
  • i'm having a meeting with patient 123, what are the meeting details?


UI_RECOMMENDATION={
  "view": "Patient",
  "action": "view",
  "params": {}
}
****************************************************************************************************
